### Step 1: Environment Setup

This notebook clones **[NVIDIA/kvpress](https://github.com/NVIDIA/kvpress)** (upstream) and installs it in editable mode. It matches **Colab’s transformers 5.x** and **`DynamicCache`** out of the box—no extra `curl` patch. Your course may link to [GabrieleSanmartino/kvpress](https://github.com/GabrieleSanmartino/kvpress); that fork is the same project line; NVIDIA `main` is usually **newer** (API fixes, optional presses like `FastKVzipPress`). If the rubric requires that exact fork URL, ask the instructor or cite “same codebase, NVIDIA upstream” in the report.

**Run Step 1b next** (small patch): on some Colab stacks, `ExpectedAttentionPress` + `model.generate` hits a bad `QuantizedCache` type check and errors with `DynamicCache` / `key_cache`. Step 1b fixes that without changing compression logic.

In [ ]:
import os
import shutil

# Ensure we are in /content before starting any operations
os.chdir('/content')

# Remove the existing kvpress directory to ensure a clean clone
if os.path.exists('kvpress'):
    shutil.rmtree('kvpress')

!git clone https://github.com/NVIDIA/kvpress.git

# Change into the cloned repository's root directory explicitly
%cd /content/kvpress

# Verify the current working directory
!pwd

# Now install the package in editable mode from its root
!pip install -e .
!pip install -q transformers accelerate bitsandbytes huggingface_hub


Cloning into 'kvpress'...
remote: Enumerating objects: 1288, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 1288 (delta 9), reused 4 (delta 2), pack-reused 1251 (from 2)
Receiving objects: 100% (1288/1288), 7.16 MiB | 34.40 MiB/s, done.
Resolving deltas: 100% (838/838), done.
/content/kvpress
/content/kvpress
Obtaining file:///content/kvpress
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kvpress (pyproject.toml) ... done
  Created wheel for kvpress: filename=kvpress-0.5.0-py3-none-any.whl size=11371 sha256=78fa5e70f6278197e9eadc37e937389c09facd4eccf3146b18a272b925e128c4
  Stored in directory: /tmp/pip-ephem-wheel-cache-8c11ktr_/wheels/2f/51/b6/70a97b3f072cbaabcdbd42cc077c7d78b5c98ae2c55d5cfa91


### Step 1b: Cache-type guard (Colab + Llama + `generate`)

Some Colab / `transformers` builds make `isinstance(past_key_values, QuantizedCache)` true even when the container is really a **`DynamicCache`** (plain `DynamicLayer` per layer). Then `BasePress.forward_hook` takes the **quantized** write path and you get **`DynamicCache` has no attribute `key_cache`**.

This cell patches kvpress to branch on **`isinstance(cache_layer, QuantizedLayer)`** (true KIVI-style KV layers) instead of the cache container type. Run once right after Step 1 (this cell imports `kvpress` and applies the patch).

In [ ]:
import torch
from transformers.cache_utils import QuantizedLayer

import kvpress.utils as _kv_utils
import kvpress.presses.base_press as _kv_base

def _extract_keys_and_values_fixed(cache, layer_idx: int):
    layer = cache.layers[layer_idx]
    if isinstance(layer, QuantizedLayer):
        return _kv_utils.dequantize_layer(layer)
    return layer.keys, layer.values


def _forward_hook_fixed(self, module, input, kwargs, output):
    hidden_states = kwargs["hidden_states"]
    cache = kwargs["past_key_values"]
    cache_layer = cache.layers[module.layer_idx]
    q_len = hidden_states.shape[1]
    if kwargs["cache_position"][-1] > q_len:
        return output
    keys, values = _extract_keys_and_values_fixed(cache, module.layer_idx)
    keys, values = self.compress(module, hidden_states, keys, values, output[1], kwargs)
    if isinstance(cache_layer, QuantizedLayer):
        cache_layer._quantized_keys = cache_layer._quantize(keys, axis=cache_layer.axis_key)
        cache_layer._quantized_values = cache_layer._quantize(values, axis=cache_layer.axis_value)
        cache_layer.keys = torch.zeros(0, dtype=keys.dtype, device=keys.device)
        cache_layer.values = torch.zeros(0, dtype=keys.dtype, device=keys.device)
        cache_layer.cumulative_length = keys.shape[2]
    else:
        cache_layer.keys = keys
        cache_layer.values = values
    return output


_kv_utils.extract_keys_and_values = _extract_keys_and_values_fixed
_kv_base.BasePress.forward_hook = _forward_hook_fixed

# KVzipPress defines its own forward_hook and still branches on isinstance(cache, QuantizedCache).
import kvpress.presses.kvzip_press as _kv_kz


def _kvzip_forward_hook_fixed(self, module, input, kwargs, output):
    hidden_states = kwargs["hidden_states"]
    cache = kwargs.get("past_key_values", None) or kwargs.get("past_key_value", None)
    cache_layer = cache.layers[module.layer_idx]
    keys, values = _extract_keys_and_values_fixed(cache, module.layer_idx)
    keys, values = self.score_kvzip(module, hidden_states, keys, values, output[1], kwargs)
    if isinstance(cache_layer, QuantizedLayer):
        cache_layer._quantized_keys = cache_layer._quantize(keys, axis=cache_layer.axis_key)
        cache_layer._quantized_values = cache_layer._quantize(values, axis=cache_layer.axis_value)
        cache_layer.keys = torch.zeros(0, dtype=keys.dtype, device=keys.device)
        cache_layer.values = torch.zeros(0, dtype=keys.dtype, device=keys.device)
        cache_layer.cumulative_length = keys.shape[2]
    else:
        cache_layer.keys = keys
        cache_layer.values = values
    return output


_kv_kz.KVzipPress.forward_hook = _kvzip_forward_hook_fixed
print("kvpress: BasePress + KVzipPress forward_hook + extract_keys_and_values patched (QuantizedLayer branch).")

### Step 2: Hugging Face Authentication

To access gated models like Llama 3.1, you need to authenticate with the Hugging Face Hub. This cell provides a login interface for your API token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Step 3: Library Optimization

This step ensures `bitsandbytes` is updated to the latest version to support optimized 4-bit and 8-bit quantization for efficient GPU memory usage.

In [ ]:
!pip install -U bitsandbytes>=0.46.1

### Step 4: Model Loading

We load the `meta-llama/Llama-3.1-8B-Instruct` model using 4-bit NormalFloat (NF4) quantization. The `device_map='auto'` setting ensures the model is automatically placed on the available GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

#load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# load 4-bit model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [2]:
filter_queries = [
    "Is the review clearly positive? (yes/no)",
    "Is the review clearly negative? (yes/no)",
    "Does the review mention excellent acting? (yes/no)",
    "Does the review mention a poor plot? (yes/no)",
    "Does the review mention great cinematography? (yes/no)",
    "Does the review mention terrible dialogue? (yes/no)",
    "Is this a rave review (extremely positive)? (yes/no)",
    "Is this a scathing review (extremely negative)? (yes/no)",
    "Does the reviewer praise the soundtrack? (yes/no)",
    "Does the reviewer criticize the special effects? (yes/no)"
]

extract_queries = [
    "Extract the sentiment of the review (positive or negative):",
    "Extract the movie title mentioned in the review:",
    "Extract one actor that is praised particularly in the review (or 'none' if no actor is praised):",
    "Extract one aspect of the movie that is criticized in the review (choose from: plot, acting, cinematography, soundtrack, special effects, dialogue, or 'none'):",
    "Extract whether the reviewer would recommend the movie based on the review (yes/no):",
    "Extract the main emotion expressed by the reviewer (e.g., joy, disappointment, anger, excitement, confusion):",
    "Extract the director's name mentioned in the review (or 'none' if not mentioned):",
    "Extract whether the review contains spoilers (yes/no):",
    "Extract whether the reviewer compares the movie to another film (yes/no):",
    "Extract the target audience implied or stated (e.g., families, children, adults, fans of action, or 'none'):"
]

all_queries = filter_queries + extract_queries

In [3]:
from kvpress import ExpectedAttentionPress, KVzipPress

# Sample review data for testing
sample_review = "The special effects were mind-blowing, but the acting of the lead actor was quite stiff."

# Select a task and compression ratio
selected_query = all_queries[0]  # Extract sentiment
compression_ratio = 0.4          # Compress to 40%

# Construct the final prompt
prompt = f"Review: {sample_review}\nTask: {selected_query}\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# --- Run Expected Attention ---
print(f"--- Testing Expected Attention (Ratio: {compression_ratio}) ---")
with ExpectedAttentionPress(compression_ratio=compression_ratio)(model):
    outputs = model.generate(**inputs, max_new_tokens=20)
    print("AI Result:", tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1])

# --- Run KVzip (class name is KVzipPress: lowercase "z" in "zip") ---
print(f"\n--- Testing KVzipPress (Ratio: {compression_ratio}) ---")
try:
    with KVzipPress(compression_ratio=compression_ratio)(model):
        outputs = model.generate(**inputs, max_new_tokens=20)
        print("AI Result:", tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1])
except Exception as e:
    print(f"KVzipPress execution failed, possible reason: {e}")

ModuleNotFoundError: No module named 'kvpress'

### Step 6: Benchmark protocol (per instructor)

- **Model**: `meta-llama/Llama-3.1-8B-Instruct`
- **Methods**: **Expected Attention** (`ExpectedAttentionPress`) vs **KVzip** (`KVzipPress`)
- **Compression ratios**: e.g. `0.2, 0.4, 0.6, 0.8, 0.9` (adjust if runs are too slow)
- **Queries**: the 10 filter-style + 10 extract-style prompts in the previous cell (plain text, not `OperatorOption` syntax)
- **Coverage**: ideally average metrics over **all** queries; if too slow, subsample a few filter and a few extract queries
- **Data**: ~1000 reviews — sample **5–10%** (e.g. `REVIEW_SAMPLE_FRAC = 0.08`). Upload a CSV to Colab and set `REVIEWS_CSV_PATH` and `REVIEW_TEXT_COLUMN` (e.g. `reviewtext`)

The code cell below sets these knobs and provides helpers; uncomment the nested loops when you are ready for a full (slow) run. **KVzip** is much heavier per prompt than Expected Attention.

In [ ]:
import random
import time
from pathlib import Path

import pandas as pd
from kvpress import ExpectedAttentionPress, KVzipPress

# Instructor-suggested compression grid (trim if too slow)
COMPRESSION_RATIOS = [0.2, 0.4, 0.6, 0.8, 0.9]
REVIEW_SAMPLE_FRAC = 0.08  # 5–10% of ~1000 reviews
MAX_NEW_TOKENS = 32

# Upload your reviews file to Colab and set path + text column name
REVIEWS_CSV_PATH = None  # e.g. Path("/content/reviews.csv")
REVIEW_TEXT_COLUMN = "reviewtext"

# If a full pass is too slow: use index lists to subsample queries (None = all)
FILTER_QUERY_INDICES = None  # e.g. [0, 1, 2, 3, 4]
EXTRACT_QUERY_INDICES = None  # e.g. [0, 1, 2, 3, 4]


def build_prompt(review_text: str, task: str) -> str:
    return f"Review: {review_text}\nTask: {task}\nAnswer:"


def decode_answer(outputs, tokenizer):
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full.split("Answer:")[-1].strip()


def run_with_press(model, tokenizer, review_text: str, task: str, press_cls, compression_ratio: float):
    prompt = build_prompt(review_text, task)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": MAX_NEW_TOKENS}
    if getattr(tokenizer, "pad_token_id", None) is not None:
        gen_kw["pad_token_id"] = tokenizer.pad_token_id
    t0 = time.perf_counter()
    with press_cls(compression_ratio=compression_ratio)(model):
        outputs = model.generate(**inputs, **gen_kw)
    elapsed = time.perf_counter() - t0
    return decode_answer(outputs, tokenizer), elapsed


def load_review_texts():
    if REVIEWS_CSV_PATH is not None and Path(REVIEWS_CSV_PATH).is_file():
        df = pd.read_csv(REVIEWS_CSV_PATH)
        texts = df[REVIEW_TEXT_COLUMN].astype(str).tolist()
    else:
        texts = [
            "The special effects were mind-blowing, but the acting felt stiff.",
            "A masterpiece of storytelling with unforgettable performances.",
        ]
    k = max(1, int(len(texts) * REVIEW_SAMPLE_FRAC))
    random.seed(42)
    return random.sample(texts, min(k, len(texts)))


def queries_for_run():
    fq = (
        [filter_queries[i] for i in FILTER_QUERY_INDICES]
        if FILTER_QUERY_INDICES is not None
        else list(filter_queries)
    )
    eq = (
        [extract_queries[i] for i in EXTRACT_QUERY_INDICES]
        if EXTRACT_QUERY_INDICES is not None
        else list(extract_queries)
    )
    return fq + eq


# --- Quick smoke test (one review, one query, one ratio) ---
_reviews = load_review_texts()
_qs = queries_for_run()
_r, _q, _cr = _reviews[0], _qs[0], 0.4
ea_ans, ea_t = run_with_press(model, tokenizer, _r, _q, ExpectedAttentionPress, _cr)
print("Smoke test ExpectedAttention:", ea_ans[:200], "| sec", round(ea_t, 3))
try:
    kz_ans, kz_t = run_with_press(model, tokenizer, _r, _q, KVzipPress, _cr)
    print("Smoke test KVzipPress:     ", kz_ans[:200], "| sec", round(kz_t, 3))
except Exception as e:
    print("Smoke test KVzipPress failed:", e)

# --- Full grid (expensive): uncomment to run ---
# rows = []
# for rev in _reviews:
#     for task in _qs:
#         for cr in COMPRESSION_RATIOS:
#             a1, t1 = run_with_press(model, tokenizer, rev, task, ExpectedAttentionPress, cr)
#             try:
#                 a2, t2 = run_with_press(model, tokenizer, rev, task, KVzipPress, cr)
#             except Exception as e:
#                 a2, t2 = f"ERR {e}", float("nan")
#             rows.append({"review": rev[:80], "task": task[:60], "ratio": cr, "ea_s": t1, "kvzip_s": t2, "ea_out": a1, "kvzip_out": a2})
# pd.DataFrame(rows).to_csv("benchmark_runs.csv", index=False)
# print("Wrote benchmark_runs.csv")